In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Computing the Conjunctive Normal Form in First Order Logic

In order to convert a formula $f$ from first order logic into a set of clauses that is satisfiable if and only if $f$ is satisfiable,
we have to perform the following steps in order:
- eliminate biconditionals and conditionals, i.e. we transform the formula into a *pure* formula,
- transform the formula into *negation normal form*,
  i.e. we push the negation symbol inwards,
- rename bound variables to avoid clashes, i.e. we transform the formula into a *rectified* formula,
- transform the formula into *prenex normal form*,
  i.e. we move the quantifiers outside,
- eliminate existential quantifiers by *skolemizing* the formula, and
- transform the formula into *conjunctive normal form* using set notation.

As in the notebook `04-CNF.ipynb` of chapter 4, we define a separate type for every intermediate form.  The type of every function then states which kind of formula it expects and which kind of formula it returns.  Hence the <samp>OCaml</samp> compiler checks that the steps are performed in the right order.

Some of these steps only work if the formula is *rectified*, where we define a formula $f$ as *rectified* if all quantifiers appearing in $f$ bind **different** variables.  For example, the formula
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall X: q(X)\bigr)$$
is **not** *rectified*, because there are two different universal quantifiers that both bind the same variable $X$.  We can rewrite this formula as a *rectified* formula by *renaming* all occurrences of $X$ that are bound by the second quantifier as follows:
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall Y: q(Y)\bigr)$$
Even if the given formula is rectified, the elimination of biconditionals produces formulas that are not rectified, because this elimination duplicates subformulas.  Therefore, the bound variables are renamed after the formula has been transformed into negation normal form.

## Auxiliary Functions

Formulas are represented as values of the type `formula` that is defined in the notebook `FOL-Parser.ipynb`.  This notebook also provides the function `parse` that converts a string into a formula, and the functions `term_to_string` and `formula_to_string` that convert terms and formulas into strings.  Our parser distinguishes variables and function symbols as follows:
- A word starting with an *upper* case letter is interpreted as a *variable*.
- A word starting with a *lower* case letter is assumed to be a *function* or *predicate symbol*.

In [ ]:
import_notebook "FOL-Parser.ipynb"

A formula in conjunctive normal form is represented in *set notation*, i.e. as a set of *clauses*, where every clause is a set of *literals*.  A literal is either an atomic formula or a negated atomic formula.  We define the following types:
- `Pos (p, [t1; ...; tn])` represents the literal $p(t_1, \cdots, t_n)$,
- `Neg (p, [t1; ...; tn])` represents the literal $\neg p(t_1, \cdots, t_n)$,
- a `clause` is a set of literals,
- a `cnf` is a set of clauses.

In TypeScript, terms and formulas are nested arrays, which have to be converted into tuples before they can be stored in a set.  In <samp>OCaml</samp>, values of variant types can be stored in an `RSet.t` directly.  These types are also used in the notebook `03-Prover.ipynb`.

In [ ]:
type literal =
  | Pos of string * term list   (* p(t1, ..., tn) *)
  | Neg of string * term list   (* ¬p(t1, ..., tn) *)

type clause = literal RSet.t
type cnf    = clause RSet.t

The function `literal_to_string` takes one argument:
- `l` is a literal.

It returns a string representing `l`, e.g. `¬p(X, f(Y))`.

The function `clause_to_string` takes one argument:
- `c` is a clause.

It returns a string representing `c`, e.g. `{¬p(X), q(X)}`.

The function `cnf_to_string` takes one argument:
- `clauses` is a set of clauses.

It returns a string representing `clauses`, where every clause is written on a line of its own.

In [ ]:
let literal_to_string (l : literal) : string =
  match l with
  | Pos (p, ts) -> term_to_string (Fun (p, ts))
  | Neg (p, ts) -> "¬" ^ term_to_string (Fun (p, ts))

let clause_to_string (c : clause) : string =
  RSet.to_string literal_to_string c

let cnf_to_string (clauses : cnf) : string =
  if RSet.is_empty clauses then
    "{}"
  else
    "{\n" ^ String.concat ",\n" (List.map (fun c -> "    " ^ clause_to_string c) (RSet.to_list clauses)) ^ "\n}"

For testing purposes, the following formula is used.  This formula specifies the notion of a *grandparent*.

In [ ]:
let s  = "∀G:∀C:(grandparent(G, C) ↔ ∃P:(parent(G, P) ∧ parent(P, C)))"
let f1 = parse s

As values of type `formula` are hard to read, we will mostly display formulas using the function `formula_to_string`.

In [ ]:
formula_to_string f1

A *substitution* $\sigma = \{ x_1 \mapsto s_1, \cdots, x_n \mapsto s_n\}$ maps variables to terms.  It is represented as a value of the type `substitution`, which is an abbreviation of the type `(string, term) RMap.t`.

The function `apply_term` takes two arguments:
- `sigma` is a substitution,
- `t` is a term.

It returns the term $t\sigma$ that results from replacing every occurrence of a variable $x_i$ in `t` by the corresponding term $s_i$.  Variables that do not occur in the domain of `sigma` are left unchanged.

In [ ]:
type substitution = (string, term) RMap.t

let rec apply_term (sigma : substitution) (t : term) : term =
  match t with
  | Var x ->
      (match RMap.find_opt x sigma with
       | Some s -> s
       | None   -> t)
  | Fun (f, ts) -> Fun (f, List.map (apply_term sigma) ts)

In [ ]:
let sigma1 : substitution = RMap.of_list [("X", parse_term "f(Y)"); ("Y", parse_term "c")]

In [ ]:
term_to_string (apply_term sigma1 (parse_term "g(X, Y, Z)"))

The function `term_variables` takes one argument:
- `t` is a term.

It returns the set of all variables that occur in `t`.

In [ ]:
let rec term_variables (t : term) : string RSet.t =
  match t with
  | Var x       -> RSet.singleton x
  | Fun (_, ts) -> RSet.unions (List.map term_variables ts)

In [ ]:
term_variables (parse_term "f(X, g(Y, X), c)")

## Eliminating Biconditionals and Conditionals

A formula is *pure* if it contains only atomic formulas, the constants `⊤` and `⊥`, the operators `¬`, `∧`, and `∨`, and quantifiers.  Pure formulas are represented by the type `Pure.t`.  Its constructors are the constructors of the type `formula` except for `Imp` and `Iff`.

As in the notebook `04-CNF.ipynb` of chapter 4, the type is defined inside the module `Pure`.  Outside of this module, its constructors are written as `Pure.Atom`, `Pure.Not`, and so on.  If we defined the constructors at the top level, they would hide the constructors of the type `formula` that have the same names.

In [ ]:
module Pure = struct
  type t =
    | Verum                          (* ⊤ *)
    | Falsum                         (* ⊥ *)
    | Atom   of string * term list   (* p(t1, ..., tn) *)
    | Not    of t                    (* ¬ *)
    | And    of t * t                (* ∧ *)
    | Or     of t * t                (* ∨ *)
    | Forall of string * t           (* ∀x: f *)
    | Exists of string * t           (* ∃x: f *)
end

The function `pure_to_formula` takes one argument:
- `f` is a pure formula.

It returns `f` as a value of the type `formula`.  As every pure formula is also a formula, this conversion is straightforward.  We need it in order to display pure formulas with the function `formula_to_string`.

In [ ]:
let rec pure_to_formula (f : Pure.t) : formula =
  match f with
  | Pure.Verum         -> Verum
  | Pure.Falsum        -> Falsum
  | Pure.Atom (p, ts)  -> Atom (p, ts)
  | Pure.Not g         -> Not (pure_to_formula g)
  | Pure.And (g, h)    -> And (pure_to_formula g, pure_to_formula h)
  | Pure.Or  (g, h)    -> Or  (pure_to_formula g, pure_to_formula h)
  | Pure.Forall (x, g) -> Forall (x, pure_to_formula g)
  | Pure.Exists (x, g) -> Exists (x, pure_to_formula g)

The function `purify` takes one argument:
- `f` is a formula from first order logic.

It returns a pure formula that is equivalent to `f`, i.e. all occurrences of the operators `↔` and `→` are eliminated.  This is done by using the following equivalences:
$$ (f \leftrightarrow g) \;\Leftrightarrow\; (f \rightarrow g) \wedge (g \rightarrow f) $$
$$ (f \rightarrow g) \;\Leftrightarrow\; (\neg f \vee g) $$
The TypeScript version of this notebook eliminates the biconditionals and the conditionals with two separate functions.  Here, `purify` does both.

**Implementation:** If `f` is a biconditional or a conditional, then `f` is first rewritten into an equivalent value of type `formula` using the corresponding equivalence.  Then `purify` is applied to the rewritten formula.  Although the rewritten formula is bigger than `f`, the recursion terminates: after at most two rewriting steps, `purify` is only called with subformulas of `f`.

As the formulas $f$ and $g$ are duplicated when a biconditional is eliminated, the resulting formula is in general not <em style="color:blue">rectified</em>.  This is taken care of by the function `rectify`, which is defined below.

In [ ]:
let rec purify (f : formula) : Pure.t =
  match f with
  | Verum         -> Pure.Verum
  | Falsum        -> Pure.Falsum
  | Atom (p, ts)  -> Pure.Atom (p, ts)
  | Not g         -> Pure.Not (purify g)
  | And (g, h)    -> Pure.And (purify g, purify h)
  | Or  (g, h)    -> Pure.Or  (purify g, purify h)
  | Imp (g, h)    -> purify (Or (Not g, h))
  | Iff (g, h)    -> purify (And (Imp (g, h), Imp (h, g)))
  | Forall (x, g) -> Pure.Forall (x, purify g)
  | Exists (x, g) -> Pure.Exists (x, purify g)

In [ ]:
formula_to_string f1

In [ ]:
let f2 = purify f1

In [ ]:
formula_to_string (pure_to_formula f2)

## Negation Normal Form

A formula is in *negation normal form* if it is pure and the negation operator `¬` is only applied to atomic formulas.  Hence, a formula in negation normal form is built from literals and the constants `⊤` and `⊥` using the operators `∧` and `∨` and quantifiers.  The type `NNF.t` represents these formulas:
- `NNF.Lit l` represents the literal `l`, i.e. either an atomic formula or a negated atomic formula.
- `NNF.t` has no constructor for negation.  Therefore, a value of type `NNF.t` can not contain any negation except for the negations of atomic formulas that are part of literals.

The type is defined inside a module for the same reason as the type `Pure.t`.

In [ ]:
module NNF = struct
  type t =
    | Lit    of literal    (* p(t1, ..., tn) or ¬p(t1, ..., tn) *)
    | Verum                (* ⊤ *)
    | Falsum               (* ⊥ *)
    | And    of t * t      (* ∧ *)
    | Or     of t * t      (* ∨ *)
    | Forall of string * t (* ∀x: f *)
    | Exists of string * t (* ∃x: f *)
end

The function `literal_to_formula` takes one argument:
- `l` is a literal.

It returns `l` as a value of the type `formula`.

The function `nnf_to_formula` takes one argument:
- `f` is a formula in negation normal form.

It returns `f` as a value of the type `formula`.  We need these functions in order to display formulas in negation normal form.

In [ ]:
let literal_to_formula (l : literal) : formula =
  match l with
  | Pos (p, ts) -> Atom (p, ts)
  | Neg (p, ts) -> Not (Atom (p, ts))

let rec nnf_to_formula (f : NNF.t) : formula =
  match f with
  | NNF.Lit l         -> literal_to_formula l
  | NNF.Verum         -> Verum
  | NNF.Falsum        -> Falsum
  | NNF.And (g, h)    -> And (nnf_to_formula g, nnf_to_formula h)
  | NNF.Or  (g, h)    -> Or  (nnf_to_formula g, nnf_to_formula h)
  | NNF.Forall (x, g) -> Forall (x, nnf_to_formula g)
  | NNF.Exists (x, g) -> Exists (x, nnf_to_formula g)

The function `nnf` takes one argument:
- `f` is a pure formula.

It returns the <em style="color:blue;">negation normal form</em> of $f$.  The auxiliary function `neg` takes the same kind of argument `f` and returns the *negation normal form* of $\neg f$.

The function $\texttt{nnf}$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(p) = p$ for all atomic formulas $p$, $\texttt{nnf}(\top) = \top$, and $\texttt{nnf}(\bot) = \bot$,</li>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) =
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) =
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\forall x: F) = \forall x: \texttt{nnf}(\texttt{F})$,</li>
    <li> $\texttt{nnf}(\exists x: F) = \exists x: \texttt{nnf}(\texttt{F})$.</li>
</ol>

The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all atomic formulas $p$,</li>
    <li> $\texttt{neg}(\top) = \bot$ and $\texttt{neg}(\bot) = \top$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\forall x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \forall x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\exists x: \neg F\bigr) \\[0.1cm]
       = & \exists x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \exists x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\forall x: F \bigr) = \exists x: \texttt{neg}(F)$.</li>
      <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\exists x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \exists x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\forall x: \neg F\bigr) \\[0.1cm]
       = & \forall x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \forall x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\exists x: F \bigr) = \forall x: \texttt{neg}(F)$.</li>
</ol>

**Implementation:** The functions `nnf` and `neg` are mutually recursive and are therefore defined together using `let rec ... and ...`.  An atomic formula $p(t_1, \cdots, t_n)$ is mapped to the literal `Pos (p, [t1; ...; tn])` by `nnf` and to the literal `Neg (p, [t1; ...; tn])` by `neg`.

In [ ]:
let rec nnf (f : Pure.t) : NNF.t =
  match f with
  | Pure.Verum         -> NNF.Verum
  | Pure.Falsum        -> NNF.Falsum
  | Pure.Atom (p, ts)  -> NNF.Lit (Pos (p, ts))
  | Pure.Not g         -> neg g
  | Pure.And (g, h)    -> NNF.And (nnf g, nnf h)
  | Pure.Or  (g, h)    -> NNF.Or  (nnf g, nnf h)
  | Pure.Forall (x, g) -> NNF.Forall (x, nnf g)
  | Pure.Exists (x, g) -> NNF.Exists (x, nnf g)

and neg (f : Pure.t) : NNF.t =
  match f with
  | Pure.Verum         -> NNF.Falsum
  | Pure.Falsum        -> NNF.Verum
  | Pure.Atom (p, ts)  -> NNF.Lit (Neg (p, ts))
  | Pure.Not g         -> nnf g
  | Pure.And (g, h)    -> NNF.Or  (neg g, neg h)
  | Pure.Or  (g, h)    -> NNF.And (neg g, neg h)
  | Pure.Forall (x, g) -> NNF.Exists (x, neg g)
  | Pure.Exists (x, g) -> NNF.Forall (x, neg g)

In [ ]:
let f3 = nnf f2

In [ ]:
formula_to_string (nnf_to_formula f3)

## Renaming Bound Variables

The formula `f3` is not rectified: the variable `P` is bound by two different quantifiers.

The function `all_variables` takes one argument:
- `f` is a formula in negation normal form.

It returns the set of all variables occurring in `f`.  This set contains both the variables occurring in terms and the variables bound by quantifiers.

In [ ]:
let rec all_variables (f : NNF.t) : string RSet.t =
  match f with
  | NNF.Lit (Pos (_, ts)) | NNF.Lit (Neg (_, ts)) -> RSet.unions (List.map term_variables ts)
  | NNF.Verum | NNF.Falsum                        -> RSet.empty
  | NNF.And (g, h) | NNF.Or (g, h)                -> RSet.union (all_variables g) (all_variables h)
  | NNF.Forall (x, g) | NNF.Exists (x, g)         -> RSet.add x (all_variables g)

In [ ]:
all_variables f3

The function `apply_literal` takes two arguments:
- `sigma` is a substitution,
- `l` is a literal.

It applies `sigma` to all terms occurring in `l`.

In [ ]:
let apply_literal (sigma : substitution) (l : literal) : literal =
  match l with
  | Pos (p, ts) -> Pos (p, List.map (apply_term sigma) ts)
  | Neg (p, ts) -> Neg (p, List.map (apply_term sigma) ts)

Below we construct a list of all upper case characters to generate new variables.  `List.init n g` returns the list `[g 0; g 1; ...; g (n-1)]`, and `String.make 1 c` turns the character `c` into a string.

In [ ]:
let ascii_uppercase : string list =
  List.init 26 (fun i -> String.make 1 (Char.chr (Char.code 'A' + i)))

The function `rectify` takes one argument:
- `f` is a formula in negation normal form.

It returns a *rectified* formula that is equivalent to `f`.  To this end, `f` is traversed from left to right.  If a quantifier binds a variable $X$ that has already been bound by a quantifier encountered earlier, then $X$ is replaced by a **new** variable, both in the quantifier and in its scope.  All other quantifiers keep their variables.  The new variables are upper case letters that do not occur in $f$.  This only works if the list `ascii_uppercase` has enough letters that do not already occur in $f$.  This approach would not be good enough for a production quality program,
but for the case of a demonstration it is sufficient.  The alternative would be to rename the variables as `X1`, `X2`, `X3`, $\cdots$, but that becomes unreadable very fast.

**Implementation:**
- `vars` is the set of all variables occurring in $f$.  As the new variables do not occur in `vars`, they can not clash with variables that are bound later in $f$ or that occur free in $f$.
- `bound` is a reference to the list of the variables that have been bound so far.  It is updated during the traversal of $f$.  Therefore, the left subformula of a conjunction or disjunction has to be processed before the right subformula.  <samp>OCaml</samp> does not specify the order in which the arguments of a constructor or a function are evaluated; in fact, they are usually evaluated from right to left.  Hence the subformulas are renamed in two separate `let` definitions.
- The local function `new_variable x` returns `x` if `x` has not been bound yet.  Otherwise, it returns the first letter that occurs neither in `vars` nor in `bound`.
- The local function `rename sigma g` renames the bound variables of a part `g` of $f$.  The substitution `sigma` maps every variable that is bound by a quantifier surrounding `g` to its (possibly new) name.

In [ ]:
let rectify (f : NNF.t) : NNF.t =
  let vars  = all_variables f in
  let bound = ref [] in
  let new_variable (x : string) : string =
    if not (List.mem x !bound) then
      x
    else
      match List.find_opt (fun v -> not (RSet.mem v vars || List.mem v !bound)) ascii_uppercase with
      | Some y -> y
      | None   -> failwith "rectify: not enough new variables"
  in
  let rec rename (sigma : substitution) (g : NNF.t) : NNF.t =
    match g with
    | NNF.Lit l              -> NNF.Lit (apply_literal sigma l)
    | NNF.Verum | NNF.Falsum -> g
    | NNF.And (g1, g2) ->
        let r1 = rename sigma g1 in   (* g1 has to be renamed before g2 *)
        let r2 = rename sigma g2 in
        NNF.And (r1, r2)
    | NNF.Or (g1, g2) ->
        let r1 = rename sigma g1 in
        let r2 = rename sigma g2 in
        NNF.Or (r1, r2)
    | NNF.Forall (x, h) ->
        let y = new_variable x in
        bound := y :: !bound;
        NNF.Forall (y, rename (RMap.add x (Var y) sigma) h)
    | NNF.Exists (x, h) ->
        let y = new_variable x in
        bound := y :: !bound;
        NNF.Exists (y, rename (RMap.add x (Var y) sigma) h)
  in
  rename RMap.empty f

In [ ]:
formula_to_string (nnf_to_formula f3)

In [ ]:
let f4 = rectify f3

In [ ]:
formula_to_string (nnf_to_formula f4)

The example formula from the beginning of this notebook is rectified as follows:

In [ ]:
formula_to_string (nnf_to_formula (rectify (nnf (purify (parse "(∀X:p(X)) ∨ (∀X:q(X))")))))

## Prenex Normal Form

The functions given below only work if all quantifiers that occur in a formula bind **different** variables, i.e. if the formulas are *rectified*.  If this assumption is not satisfied, then they will produce <u>garbage</u>.  This is the reason why we have renamed the bound variables in the previous section.

A *quantifier prefix* is a list of quantifiers together with the variables they bind, i.e. a list of the form
$$ [Q_1\,x_1; \cdots; Q_n\,x_n] $$
Here, the $Q_i$ denote quantifiers, i.e. we have $Q_i \in \{\forall, \exists\}$, while the $x_i$ are variables.  An element $Q_i\,x_i$ of a quantifier prefix is represented as a value of the type `quantifier`: `All x` represents $\forall x$ and `Ex x` represents $\exists x$.

In [ ]:
type quantifier =
  | All of string   (* ∀x *)
  | Ex  of string   (* ∃x *)

The *matrix* of a formula in prenex normal form is the part that remains when the quantifier prefix is removed.  Hence, the matrix is a formula in negation normal form that does not contain any quantifiers.  The type `Matrix.t` represents these formulas.  Its constructors are the constructors of the type `NNF.t` except for `Forall` and `Exists`.

A formula in prenex normal form is represented as a pair `(qs, m)`, where `qs` is a quantifier prefix and `m` is a matrix.

In [ ]:
module Matrix = struct
  type t =
    | Lit    of literal   (* p(t1, ..., tn) or ¬p(t1, ..., tn) *)
    | Verum               (* ⊤ *)
    | Falsum              (* ⊥ *)
    | And    of t * t     (* ∧ *)
    | Or     of t * t     (* ∨ *)
end

The function `matrix_to_formula` takes one argument:
- `m` is a matrix.

It returns `m` as a value of the type `formula`.  We need this function in order to display matrices.

In [ ]:
let rec matrix_to_formula (m : Matrix.t) : formula =
  match m with
  | Matrix.Lit l      -> literal_to_formula l
  | Matrix.Verum      -> Verum
  | Matrix.Falsum     -> Falsum
  | Matrix.And (g, h) -> And (matrix_to_formula g, matrix_to_formula h)
  | Matrix.Or  (g, h) -> Or  (matrix_to_formula g, matrix_to_formula h)

The function `merge_quantifiers` takes two arguments:
- `q1` is a quantifier prefix,
- `q2` is a quantifier prefix.

It merges them into a new quantifier prefix such that the relative order of the quantifiers remains the same, i.e. if both $Q_1\,x_1$ and $Q_2\,x_2$ occur in `q1` and $Q_1\,x_1$ occurs before $Q_2\,x_2$, then $Q_1\,x_1$ will occur before $Q_2\,x_2$ in the result.  Existential quantifiers are extracted first, since this results in Skolem functions with fewer arguments.

In [ ]:
let rec merge_quantifiers (q1 : quantifier list) (q2 : quantifier list) : quantifier list =
  match q1, q2 with
  | [], _              -> q2
  | _, []              -> q1
  | Ex x :: rest, _    -> Ex x :: merge_quantifiers rest q2   (* extract existential quantifiers first *)
  | _, Ex y :: rest    -> Ex y :: merge_quantifiers q1 rest
  | q :: rest, _       -> q :: merge_quantifiers rest q2

In [ ]:
merge_quantifiers [All "X"; Ex "Y"] [Ex "U"; All "V"]

The function `extract_quantifiers` takes one argument:
- `f` is a rectified formula in negation normal form.

It returns a pair `(qs, m)`, where `qs` is a quantifier prefix and `m` is the <em style="color:blue;">matrix</em> of the formula $f$.  The matrix of a formula is the part that remains when all quantifiers have been extracted.  Hence, the pair `(qs, m)` represents the *prenex normal form* of `f`.  Extracting the quantifiers of the subformulas of a conjunction or a disjunction is only correct because `f` is rectified.

In [ ]:
let rec extract_quantifiers (f : NNF.t) : quantifier list * Matrix.t =
  match f with
  | NNF.Lit l      -> ([], Matrix.Lit l)
  | NNF.Verum      -> ([], Matrix.Verum)
  | NNF.Falsum     -> ([], Matrix.Falsum)
  | NNF.And (g, h) ->
      let (qg, gm) = extract_quantifiers g in
      let (qh, hm) = extract_quantifiers h in
      (merge_quantifiers qg qh, Matrix.And (gm, hm))
  | NNF.Or (g, h) ->
      let (qg, gm) = extract_quantifiers g in
      let (qh, hm) = extract_quantifiers h in
      (merge_quantifiers qg qh, Matrix.Or (gm, hm))
  | NNF.Forall (x, g) ->
      let (qg, gm) = extract_quantifiers g in
      (All x :: qg, gm)
  | NNF.Exists (x, g) ->
      let (qg, gm) = extract_quantifiers g in
      (Ex x :: qg, gm)

In [ ]:
formula_to_string (nnf_to_formula f4)

In [ ]:
let (qs, f5) = extract_quantifiers f4

In [ ]:
formula_to_string (matrix_to_formula f5)

The function `attach_quantifiers` takes two arguments:
- `qs` is a quantifier prefix,
- `m` is a matrix.

It combines the quantifiers `qs` and the matrix `m` into a quantified formula of type `formula`.  Since we represent the prenex normal form as a pair, we need this function only in order to display the prenex normal form.

In [ ]:
let rec attach_quantifiers (qs : quantifier list) (m : Matrix.t) : formula =
  match qs with
  | []            -> matrix_to_formula m
  | All x :: rest -> Forall (x, attach_quantifiers rest m)
  | Ex  x :: rest -> Exists (x, attach_quantifiers rest m)

In [ ]:
qs

In [ ]:
formula_to_string (attach_quantifiers qs f5)

## Skolemization (Eliminating Existential Quantifiers)

The reference `skolem_counter` is a global variable that is needed to create unique Skolem functions.  The function `skolem_function` takes the argument `()` and returns a new name for a Skolem function every time it is called.

In [ ]:
let skolem_counter : int ref = ref 0

let skolem_function () : string =
  incr skolem_counter;
  "sk" ^ string_of_int !skolem_counter

The function `apply_matrix` takes two arguments:
- `sigma` is a substitution,
- `m` is a matrix.

It applies `sigma` to all terms occurring in `m`.  As a matrix does not contain quantifiers, there are no bound variables that would have to be taken care of.

In [ ]:
let rec apply_matrix (sigma : substitution) (m : Matrix.t) : Matrix.t =
  match m with
  | Matrix.Lit l                 -> Matrix.Lit (apply_literal sigma l)
  | Matrix.Verum | Matrix.Falsum -> m
  | Matrix.And (g, h)            -> Matrix.And (apply_matrix sigma g, apply_matrix sigma h)
  | Matrix.Or  (g, h)            -> Matrix.Or  (apply_matrix sigma g, apply_matrix sigma h)

The function `skolemize` takes three arguments:
- `qs` is a quantifier prefix,
- `m` is a matrix, i.e. the pair `(qs, m)` is a formula in <em style="color:blue">prenex normal form</em>,
- `vs` is a list of variables that are universally quantified.

It <em style="color:blue">skolemizes</em> the formula `(qs, m)`, i.e. it replaces all existentially quantified variables by appropriate <em style="color:blue">Skolem functions</em>.  The arguments of these Skolem functions are the variables from `vs`.  When `skolemize` is called initially, `vs` is the empty list.  During the recursion, `vs` collects the universally quantified variables that precede the current quantifier.  The result is a matrix: In contrast to the TypeScript version, the universal quantifiers are dropped, since the variables occurring in clauses are implicitly universally quantified anyway.

For skolemization to work correctly, we have to assume that
<font size="4" style="color:darkgreen; size:125%">the formula does not contain free variables</font>!

In [ ]:
let rec skolemize (qs : quantifier list) (m : Matrix.t) (vs : string list) : Matrix.t =
  match qs with
  | [] -> m
  | Ex x :: rest ->
      let t = Fun (skolem_function (), List.map (fun v -> Var v) vs) in
      skolemize rest (apply_matrix (RMap.singleton x t) m) vs
  | All x :: rest ->
      skolemize rest m (vs @ [x])

In [ ]:
formula_to_string (attach_quantifiers qs f5)

In [ ]:
let f6 = skolemize qs f5 []

In [ ]:
formula_to_string (matrix_to_formula f6)

## Conversion to Clauses

The function `cnf` takes one argument:
- `m` is a matrix that results from <em style="color:blue">skolemization</em>, i.e. a formula in <em style="color:blue">negation normal form</em> that does not contain quantifiers.

It returns the <em style="color:blue">conjunctive normal form</em> of `m` in <em style="color:blue">set notation</em>.  This works the same way as in propositional logic: For a fixed clause `c` of `gs`, the expression `RSet.map (RSet.union c) hs` computes the set of all clauses $c \cup d$ where $d$ is a clause of `hs`, and `RSet.flat_map` computes the union of these sets for all clauses `c` of `gs`.  Since every value of the type `Matrix.t` is in negation normal form and does not contain quantifiers, the pattern match covers all possible cases and `cnf` can not fail.

In [ ]:
let rec cnf (m : Matrix.t) : cnf =
  match m with
  | Matrix.Lit l      -> RSet.singleton (RSet.singleton l)
  | Matrix.Verum      -> RSet.empty
  | Matrix.Falsum     -> RSet.singleton RSet.empty
  | Matrix.And (g, h) -> RSet.union (cnf g) (cnf h)
  | Matrix.Or  (g, h) ->
      let gs = cnf g
      and hs = cnf h in
      RSet.flat_map (fun c -> RSet.map (RSet.union c) hs) gs

In [ ]:
formula_to_string (matrix_to_formula f6)

In [ ]:
let f7 = cnf f6

In [ ]:
print_endline (cnf_to_string f7)

## Putting Everything Together

The function `normalize` takes one argument:
- `f` is a formula from first order logic that **must not** contain free variables.

It transforms $f$ into a set of first order clauses that is satisfiable if and only if `f` is satisfiable.

The types of the functions ensure that the steps are performed in the right order: `purify` maps a `formula` to a `Pure.t`, `nnf` maps a `Pure.t` to an `NNF.t`, `rectify` maps an `NNF.t` to a rectified `NNF.t`, `extract_quantifiers` maps an `NNF.t` to a quantifier prefix and a `Matrix.t`, `skolemize` returns a `Matrix.t`, and `cnf` maps a `Matrix.t` to a set of clauses.

In [ ]:
let normalize (f : formula) : cnf =
  let f1       = purify f in
  let f2       = nnf f1 in
  let f3       = rectify f2 in
  let (qs, f4) = extract_quantifiers f3 in
  let f5       = skolemize qs f4 [] in
  cnf f5

In [ ]:
print_endline (cnf_to_string (normalize f1))

The function `test` takes one argument:
- `s` is a string representing a formula without free variables.

It computes the clauses of this formula and prints them.

In [ ]:
let test (s : string) : unit =
  let f = parse s in
  Printf.printf "The CNF of %s is:\n%s\n" s (cnf_to_string (normalize f))

In [ ]:
test s

In [ ]:
test "¬(∃Y:∀X:p(X,Y)→∀U:∃V:p(U,V))"

The following formula is satisfiable: in a structure with two elements, where the predicate $p$ holds for exactly one of them, both conjuncts are true.  Hence, its set of clauses has to be satisfiable, too.  When `↔` is eliminated, the formula $\forall X: p(X)$ is duplicated.  If the variable $X$ in the second copy were renamed to a variable that already occurs elsewhere in the formula, e.g. to $A$, then the prenex normal form would identify this variable with the variable $A$ that is bound by $\exists A$.  We would then get the unsatisfiable set of clauses $\bigl\{\{p(c)\}, \{\neg p(c)\}\bigr\}$ for a single Skolem constant $c$.  This is why `rectify` only uses variables that do not occur in the formula.

In [ ]:
test "(∃A:p(A)) ∧ (⊥ ↔ ∀X:p(X))"